# High-Sensitivity Eulerian Respiratory Deconvolution

This interactive notebook demonstrates the **Pose-Anchored Dual-Domain Eulerian Optical Deconvolution** pipeline. It features high-gain optical transillumination/schlieren sensitivity using Motion Band Relief, Edge Contrast Sensitivity, Dynamic Gamma, and High Anti-Blooming $\\tau$ clamping.

Adjust the parameters interactively using ipywidgets to optimize the visibility of subtle respiratory motion.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import math
import collections

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

In [ ]:
class RespirationConfig:
    def __init__(self):
        self.patch_width = 240
        self.patch_height = 180
        self.alpha = 35.0          # Eulerian amplification gain
        self.artifact_gate = 0.22  # Motion energy threshold
        self.soft_clamp = 8.0      # Tanh saturation ceiling (tau)
        self.gamma_fast = 0.20     # Fast IIR pole
        self.gradient_scale = 1.2  # Edge Contrast Sensitivity
        self.coherence_th = 0.08   # Structure tensor edge coherence
        self.gate_mode = 0         # 0=Soft AGC

class EulerianEngine:
    def __init__(self, width, height):
        self.w = width
        self.h = height
        self.iir_slow = np.zeros((height, width), dtype=np.float32)
        self.iir_fast = np.zeros((height, width), dtype=np.float32)
        self.is_initialized = False

    def reset(self):
        self.is_initialized = False

    def process(self, patch_bgr, cfg):
        gray = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
        if not self.is_initialized or self.iir_slow.shape != gray.shape:
            self.iir_slow = gray.copy()
            self.iir_fast = gray.copy()
            self.is_initialized = True
            empty_coh = np.zeros_like(gray)
            neutral_relief = np.full_like(patch_bgr, 128)
            return patch_bgr.copy(), neutral_relief, empty_coh, 0.0, 0.0, False

        gamma_slow = cfg.gamma_fast * 0.15
        self.iir_slow += gamma_slow * (gray - self.iir_slow)
        self.iir_fast += cfg.gamma_fast * (gray - self.iir_fast)
        bandpass = self.iir_fast - self.iir_slow

        g_scale = cfg.gradient_scale * 0.125
        ix = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3) * g_scale
        iy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3) * g_scale

        jxx = cv2.boxFilter(ix * ix, -1, (3, 3))
        jyy = cv2.boxFilter(iy * iy, -1, (3, 3))
        jxy = cv2.boxFilter(ix * iy, -1, (3, 3))

        diff = jxx - jyy
        trace = jxx + jyy
        num = (diff * diff) + (4.0 * jxy * jxy)
        den = (trace * trace) + 1e-4
        coherence = np.clip(num / den, 0.0, 1.0)
        coherence_mask = np.where(coherence >= cfg.coherence_th, coherence, 0.0)

        motion_energy = float(np.mean(np.abs(bandpass)))
        is_gated = (cfg.gate_mode != 2) and (motion_energy > cfg.artifact_gate)

        effective_alpha = cfg.alpha
        if is_gated:
            if cfg.gate_mode == 1:
                effective_alpha = 0.0
            else:
                excess = motion_energy / max(1e-4, cfg.artifact_gate)
                effective_alpha = cfg.alpha / (1.0 + excess * excess)

        tau = max(1.0, cfg.soft_clamp)
        clamped_motion = tau * np.tanh(bandpass / tau)
        magnified_delta = effective_alpha * coherence_mask * clamped_motion

        out_bgr = patch_bgr.astype(np.float32)
        out_bgr[:, :, 0] += magnified_delta * 1.15
        out_bgr[:, :, 1] += magnified_delta * 0.95
        out_bgr[:, :, 2] += magnified_delta
        out_bgr = np.clip(out_bgr, 0, 255).astype(np.uint8)

        diff_vis = np.clip(128.0 + clamped_motion * 18.0, 0, 255).astype(np.uint8)
        relief_bgr = cv2.cvtColor(diff_vis, cv2.COLOR_GRAY2BGR)

        motion_scalar = float(np.mean(coherence_mask * clamped_motion))
        return out_bgr, relief_bgr, coherence, motion_scalar, motion_energy, is_gated

In [ ]:
# ==============================================================================
# 3. Select Input Video
# ==============================================================================
# Use the dropdown to dynamically select which video to process.
import glob
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
import cv2
import numpy as np

def load_video_frames(video_path, max_frames=300, patch_size=(240, 180)):
    cap = cv2.VideoCapture(video_path)
    frames = []
    if not cap.isOpened():
        print(f"Could not open {video_path}")
        return frames
    
    count = 0
    while count < max_frames:
        ret, frame = cap.read()
        if not ret: break
        h, w = frame.shape[:2]
        cx, cy = w//2, h//2
        rw, rh = int(h*0.6), int(h*0.45)
        crop = frame[cy-rh//2:cy+rh//2, cx-rw//2:cx+rw//2]
        if crop.size > 0:
            frames.append(cv2.resize(crop, patch_size))
        count += 1
    cap.release()
    return frames

# Find available videos in the videos directory and root
video_files = glob.glob("../videos/*.mp4") + glob.glob("../videos/*.avi") + glob.glob("../*.mp4")
video_options = {os.path.basename(f): f for f in video_files}
video_options["Synthetic Breathing Data (Fallback)"] = "SYNTHETIC"

video_dropdown = widgets.Dropdown(
    options=video_options,
    description='Select Video:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)
load_button = widgets.Button(description="Load Frames", button_style="info")
output_loader = widgets.Output()

frames = []

def on_load_clicked(b):
    global frames
    with output_loader:
        output_loader.clear_output()
        selection = video_dropdown.value
        if selection == "SYNTHETIC":
            print("Generating synthetic breathing data...")
            frames = []
            for i in range(150):
                base = np.linspace(100, 150, 240).reshape(1, -1).repeat(180, axis=0)
                breath = 5.0 * np.sin(2 * np.pi * i / 30.0)
                patch = np.clip(base + breath, 0, 255).astype(np.uint8)
                patch_bgr = cv2.cvtColor(patch, cv2.COLOR_GRAY2BGR)
                cv2.rectangle(patch_bgr, (100, 50), (140, 130), (50,50,50), -1)
                frames.append(patch_bgr)
            print("Loaded 150 synthetic frames.")
        else:
            print(f"Loading frames from {selection}... Please wait.")
            frames = load_video_frames(selection)
            print(f"Loaded {len(frames)} frames successfully. You can now run the cells below.")

load_button.on_click(on_load_clicked)
display(widgets.HBox([video_dropdown, load_button]), output_loader)

# Pre-load the first available video so the variables exist immediately
if video_files:
    frames = load_video_frames(video_files[0])
    print(f"Automatically pre-loaded: {os.path.basename(video_files[0])} ({len(frames)} frames).")
else:
    on_load_clicked(None)


In [ ]:
def explore_eulerian(frame_idx=0, alpha=35.0, gradient_scale=1.2, gamma_fast=0.20, soft_clamp=8.0, coherence_th=0.08):
    if not frames:
        print("No frames available.")
        return
    
    cfg = RespirationConfig()
    cfg.alpha = alpha
    cfg.gradient_scale = gradient_scale
    cfg.gamma_fast = gamma_fast
    cfg.soft_clamp = soft_clamp
    cfg.coherence_th = coherence_th
    
    engine = EulerianEngine(240, 180)
    
    # We need to process from frame 0 up to frame_idx to build the temporal IIR state
    mag_patch, relief_patch, coherence_map, _, _, _ = None, None, None, None, None, None
    for i in range(frame_idx + 1):
        mag_patch, relief_patch, coherence_map, scalar, energy, gated = engine.process(frames[i], cfg)
        
    fig, axs = plt.subplots(2, 2, figsize=(12, 9))
    axs = axs.flatten()
    
    # 1. Original
    axs[0].imshow(cv2.cvtColor(frames[frame_idx], cv2.COLOR_BGR2RGB))
    axs[0].set_title("Original Patch")
    axs[0].axis("off")
    
    # 2. Color EVM (Magnified)
    axs[1].imshow(cv2.cvtColor(mag_patch, cv2.COLOR_BGR2RGB))
    axs[1].set_title(f"Color EVM (Gain: {alpha}x)")
    axs[1].axis("off")
    
    # 3. Motion Band Relief
    axs[2].imshow(cv2.cvtColor(relief_patch, cv2.COLOR_BGR2RGB))
    axs[2].set_title(f"Motion Band Relief (tau={soft_clamp})")
    axs[2].axis("off")
    
    # 4. Coherence Map
    im = axs[3].imshow(coherence_map, cmap="viridis", vmin=0, vmax=1)
    axs[3].set_title("Structure Tensor Coherence")
    axs[3].axis("off")
    plt.colorbar(im, ax=axs[3], shrink=0.8)
    
    plt.tight_layout()
    plt.show()

widgets.interact(
    explore_eulerian, 
    frame_idx=widgets.IntSlider(min=0, max=len(frames)-1, step=1, value=min(30, len(frames)-1), description="Frame:"),
    alpha=widgets.FloatSlider(min=1.0, max=80.0, step=1.0, value=35.0, description="Gain (α):"),
    gradient_scale=widgets.FloatSlider(min=0.5, max=3.0, step=0.1, value=1.2, description="Contrast:"),
    gamma_fast=widgets.FloatSlider(min=0.02, max=0.50, step=0.02, value=0.20, description="Gamma (γ):"),
    soft_clamp=widgets.FloatSlider(min=1.0, max=30.0, step=1.0, value=8.0, description="Tau (τ):"),
    coherence_th=widgets.FloatSlider(min=0.0, max=0.40, step=0.02, value=0.08, description="Coherence:")
);

In [ ]:
# ==============================================================================
# 5. Statistical Data Distribution & Spectral Analysis
# ==============================================================================
# This section mixes the Eulerian processing with the Data Distribution analysis.
# It computes the full temporal waveform, plots the physiological data distribution (KDE), 
# and extracts the Respiration Rate (RPM) via Power Spectral Density (PSD).

import seaborn as sns
from scipy import signal
import matplotlib.pyplot as plt

def analyze_data_distribution(alpha=35.0, soft_clamp=8.0, gamma_fast=0.20):
    if not frames:
        print("No frames available.")
        return
        
    cfg = RespirationConfig()
    cfg.alpha = alpha
    cfg.soft_clamp = soft_clamp
    cfg.gamma_fast = gamma_fast
    
    engine = EulerianEngine(frames[0].shape[1], frames[0].shape[0])
    
    scalars = []
    coherences = []
    
    for frame in frames:
        _, _, coherence_map, scalar, _, _ = engine.process(frame, cfg)
        scalars.append(scalar)
        coherences.append(np.mean(coherence_map))
        
    scalars = np.array(scalars)
    
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 2, height_ratios=[1.5, 1, 1])
    
    # 1. Temporal Waveform
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(scalars, color='royalblue', linewidth=2, label="Motion Signal S(t)")
    ax1.set_title("Eulerian Extracted Respiratory Waveform", fontweight='bold')
    ax1.set_xlabel("Frame Index")
    ax1.set_ylabel("Amplitude")
    ax1.legend()
    ax1.grid(True, linestyle="--", alpha=0.6)
    
    # 2. Data Distribution: Motion Amplitude Histogram & KDE
    ax2 = fig.add_subplot(gs[1, 0])
    sns.histplot(scalars, kde=True, color="seagreen", ax=ax2, stat="density", bins=30, alpha=0.4)
    ax2.set_title("Motion Amplitude Distribution (KDE)", fontweight='bold')
    ax2.set_xlabel("Amplitude")
    ax2.set_ylabel("Density")
    
    # 3. Spectral Analysis (PSD for RPM)
    ax3 = fig.add_subplot(gs[1, 1])
    fps = 30.0
    # Detrend the signal to remove DC bias for cleaner FFT
    detrended = signal.detrend(scalars)
    freqs, psd = signal.welch(detrended, fps, nperseg=min(256, len(scalars)))
    
    rpms = freqs * 60.0
    valid_idx = (rpms >= 4) & (rpms <= 45)  # Physiological range 4-45 RPM
    
    if np.any(valid_idx):
        peak_rpm = rpms[valid_idx][np.argmax(psd[valid_idx])]
        ax3.plot(rpms[valid_idx], psd[valid_idx], color="firebrick", lw=2)
        ax3.axvline(peak_rpm, color="black", linestyle="--", label=f"Peak Respiration: {peak_rpm:.1f} RPM")
        ax3.fill_between(rpms[valid_idx], psd[valid_idx], color="firebrick", alpha=0.2)
        ax3.set_title("Power Spectral Density (Frequency Analysis)", fontweight='bold')
        ax3.set_xlabel("Frequency (RPM)")
        ax3.set_ylabel("Power")
        ax3.legend()
    else:
        ax3.text(0.5, 0.5, "Insufficient data for PSD", ha='center')
        
    # 4. Coherence Distribution
    ax4 = fig.add_subplot(gs[2, :])
    sns.histplot(coherences, kde=True, color="darkorange", ax=ax4, bins=30, alpha=0.4)
    ax4.set_title("Structure Tensor Coherence Distribution", fontweight='bold')
    ax4.set_xlabel("Mean Spatial Coherence (Confidence)")
    
    plt.tight_layout()
    plt.show()

widgets.interact_manual(
    analyze_data_distribution,
    alpha=widgets.FloatSlider(min=1.0, max=80.0, step=1.0, value=35.0, description="Gain (α):"),
    soft_clamp=widgets.FloatSlider(min=1.0, max=30.0, step=1.0, value=8.0, description="Tau (τ):"),
    gamma_fast=widgets.FloatSlider(min=0.02, max=0.50, step=0.02, value=0.20, description="Gamma (γ):")
);


In [ ]:
# ==============================================================================
# 4. Fluidity Visualization & MP4 Video Export
# ==============================================================================
# Generate a video of the enhanced output and display it inline.
# This makes it easy to visualize continuous motion (fluidity) and save MP4s.

from IPython.display import Video, display, HTML
import cv2
import os
import ipywidgets as widgets

def generate_and_play_video(alpha=35.0, soft_clamp=8.0, gamma_fast=0.20):
    if not frames:
        print("No frames available to process.")
        return
        
    out_file = "eulerian_fluidity_preview.mp4"
    h, w = frames[0].shape[:2]
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(out_file, fourcc, 30.0, (w * 2, h))
    
    cfg = RespirationConfig()
    cfg.alpha = alpha
    cfg.soft_clamp = soft_clamp
    cfg.gamma_fast = gamma_fast
    
    engine = EulerianEngine(w, h)
    
    print(f"Processing {len(frames)} frames into video...")
    for i, frame in enumerate(frames):
        mag_patch, relief_patch, _, _, _, _ = engine.process(frame, cfg)
        
        orig_disp = frame.copy()
        cv2.putText(orig_disp, "ORIGINAL", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
        cv2.putText(relief_patch, f"MOTION RELIEF (Gain={alpha}x)", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,168), 2)
        
        combined = np.hstack((orig_disp, relief_patch))
        writer.write(combined)
        
    writer.release()
    
    display(HTML(f"<b>✅ Video Exported Successfully!</b> <br> "
                 f"File saved to: <a href='{out_file}' target='_blank'>{out_file}</a><br>"
                 f"<i>(If it doesn't auto-play below, open the MP4 file locally in VLC/Windows Media Player)</i>"))
                 
    display(Video(out_file, embed=True, html_attributes="controls autoplay loop", width=600))

widgets.interact_manual(
    generate_and_play_video,
    alpha=widgets.FloatSlider(min=1.0, max=80.0, step=1.0, value=35.0, description="Gain (α):"),
    soft_clamp=widgets.FloatSlider(min=1.0, max=30.0, step=1.0, value=8.0, description="Tau (τ):"),
    gamma_fast=widgets.FloatSlider(min=0.02, max=0.50, step=0.02, value=0.20, description="Gamma (γ):")
);
